In [23]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

def download_day_ahead_prices(start_date="2019-01-01", end_date=None):
    """
    从SMARD API下载Day-ahead prices数据
    """
    if end_date is None:
        end_date = datetime.now().strftime("%Y-%m-%d")
    
    base_url = "https://www.smard.de/app/chart_data"
    filter_id = "4169"  # * `4169` - Marktpreis: Deutschland/Luxemburg
    region = "DE"
    resolution = "quarterhour"
    
    # 1. 获取所有可用时间戳
    index_url = f"{base_url}/{filter_id}/{region}/index_{resolution}.json"
    print(f"获取时间戳列表: {index_url}")
    
    try:
        response = requests.get(index_url, timeout=30)
        response.raise_for_status()
        timestamp_data = response.json()
        all_timestamps = timestamp_data.get('timestamps', [])
        
        print(f"找到 {len(all_timestamps)} 个时间戳")
        
    except Exception as e:
        print(f"获取时间戳失败: {e}")
        return None
    
    # 2. 过滤2019年至今的时间戳
    start_ts = int(pd.to_datetime(start_date).timestamp() * 1000)
    end_ts = int(pd.to_datetime(end_date).timestamp() * 1000)
    
    # 找到所有在时间范围内的时间戳
    relevant_timestamps = [ts for ts in all_timestamps if ts <= end_ts]
    
    # 找到最接近start_date的时间戳（可能略早）
    closest_idx = 0
    for i, ts in enumerate(relevant_timestamps):
        if ts >= start_ts:
            closest_idx = max(0, i - 1)  # 往前多取一个包
            break
    else:
        # 如果所有时间戳都早于start_ts，取最后一个
        closest_idx = len(relevant_timestamps) - 1
    
    relevant_timestamps = relevant_timestamps[closest_idx:]
    
    print(f"筛选出 {len(relevant_timestamps)} 个相关时间戳")
    if relevant_timestamps:
        earliest = pd.to_datetime(relevant_timestamps[0], unit='ms')
        latest = pd.to_datetime(relevant_timestamps[-1], unit='ms')
        print(f"时间戳范围: {earliest} 到 {latest}")
    
    # 3. 分批下载数据（避免超时）
    all_data = []
    batch_size = 50  # 每批50个时间戳
    
    for i in range(0, len(relevant_timestamps), batch_size):
        batch_timestamps = relevant_timestamps[i:i+batch_size]
        print(f"下载批次 {i//batch_size + 1}")
        
        for ts in batch_timestamps:
            data_url = f"{base_url}/{filter_id}/{region}/{filter_id}_{region}_{resolution}_{ts}.json"
            
            try:
                response = requests.get(data_url, timeout=30)
                response.raise_for_status()
                data = response.json()
                
                # 解析数据 - series格式: [[timestamp, value], ...]
                series = data.get('series', [])
                
                for item in series:
                    if isinstance(item, list) and len(item) >= 2:
                        ts_val, price_val = item[0], item[1]
                        all_data.append({
                            'timestamp': ts_val,
                            'price_eur_mwh': price_val
                        })
                
            except Exception as e:
                print(f"下载时间戳 {ts} 失败: {e}")
                continue
            
        # time.sleep(1)  # 避免请求过于频繁
    
    # 4. 转换为DataFrame
    df = pd.DataFrame(all_data)
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print(f"成功下载 {len(df)} 条价格记录")
    print(f"时间范围: {df['datetime'].min()} 到 {df['datetime'].max()}")
    
    return df

# 使用示例
df_prices = download_day_ahead_prices("2019-01-01", "2026-04-05")
if df_prices is not None:
    # 保存为CSV
    df_prices.to_csv("Day-ahead_prices_20190101_20260405.csv", index=False)

获取时间戳列表: https://www.smard.de/app/chart_data/4169/DE/index_quarterhour.json
找到 393 个时间戳
筛选出 379 个相关时间戳
时间戳范围: 2018-12-30 23:00:00 到 2026-03-29 22:00:00
下载批次 1


下载批次 2
下载批次 3
下载批次 4
下载批次 5
下载批次 6
下载批次 7
下载批次 8
成功下载 254684 条价格记录
时间范围: 2018-12-30 23:00:00 到 2026-04-05 21:45:00


In [24]:
# 读取CSV文件
df = pd.read_csv('/workspaces/Energy-Monitor-Germany/notebooks/Day-ahead_prices_20190101_20260405.csv')

# 查看前几行
print(df.head())
print(f"\n数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n数据类型:\n{df.dtypes}")

       timestamp  price_eur_mwh             datetime
0  1546210800000          50.94  2018-12-30 23:00:00
1  1546211700000          50.94  2018-12-30 23:15:00
2  1546212600000          50.94  2018-12-30 23:30:00
3  1546213500000          50.94  2018-12-30 23:45:00
4  1546214400000          49.57  2018-12-31 00:00:00

数据形状: (254684, 3)

列名: ['timestamp', 'price_eur_mwh', 'datetime']

数据类型:
timestamp          int64
price_eur_mwh    float64
datetime             str
dtype: object


In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

def download_actual_generation(start_date="2019-01-01", end_date=None):
    """
    从SMARD API下载Actual generation数据（所有能源类型）
    """
    import requests
    import pandas as pd
    from datetime import datetime, timedelta
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    if end_date is None:
        end_date = datetime.now().strftime("%Y-%m-%d")
    
    base_url = "https://www.smard.de/app/chart_data"
    region = "DE"
    resolution = "quarterhour"
    
    # 能源类型映射：filter_id -> 列名
    energy_types = {
        '4066': 'biomass',
        '1226': 'hydropower', 
        '1225': 'wind_offshore',
        '4067': 'wind_onshore',
        '4068': 'photovoltaics',
        '1228': 'other_renewable',
        '1224': 'nuclear',
        '1223': 'lignite',
        '4069': 'hard_coal',
        '4071': 'fossil_gas',
        '4070': 'hydro_pumped_storage',
        '1227': 'other_conventional'
    }
    
    print(f"开始下载发电数据: {start_date} 到 {end_date}")
    
    # 1. 获取时间戳列表（使用第一个能源类型作为代表）
    first_filter = list(energy_types.keys())[0]
    index_url = f"{base_url}/{first_filter}/{region}/index_{resolution}.json"
    print(f"获取时间戳列表: {index_url}")
    
    try:
        response = requests.get(index_url, timeout=30)
        response.raise_for_status()
        timestamp_data = response.json()
        all_timestamps = timestamp_data.get('timestamps', [])
        
        print(f"找到 {len(all_timestamps)} 个时间戳")
        
    except Exception as e:
        print(f"获取时间戳失败: {e}")
        return None
    
    # 2. 过滤时间戳
    start_ts = int(pd.to_datetime(start_date).timestamp() * 1000)
    end_ts = int(pd.to_datetime(end_date).timestamp() * 1000)
    
    # 找到所有在时间范围内的时间戳
    relevant_timestamps = [ts for ts in all_timestamps if ts <= end_ts]
    
    # 找到最接近start_date的时间戳（可能略早）
    closest_idx = 0
    for i, ts in enumerate(relevant_timestamps):
        if ts >= start_ts:
            closest_idx = max(0, i - 1)  # 往前多取一个包
            break
    else:
        # 如果所有时间戳都早于start_ts，取最后一个
        closest_idx = len(relevant_timestamps) - 1
    
    relevant_timestamps = relevant_timestamps[closest_idx:]
    
    print(f"筛选出 {len(relevant_timestamps)} 个相关时间戳")
    if relevant_timestamps:
        earliest = pd.to_datetime(relevant_timestamps[0], unit='ms')
        latest = pd.to_datetime(relevant_timestamps[-1], unit='ms')
        print(f"时间戳范围: {earliest} 到 {latest}")
    
    # 3. 并行下载所有能源类型的数据
    def download_energy_type(filter_id, col_name):
        """下载单个能源类型的数据"""
        print(f"开始下载 {col_name} (filter: {filter_id})")
        energy_data = []
        
        # 分批下载
        batch_size = 50
        for i in range(0, len(relevant_timestamps), batch_size):
            batch_timestamps = relevant_timestamps[i:i+batch_size]
            
            for ts in batch_timestamps:
                data_url = f"{base_url}/{filter_id}/{region}/{filter_id}_{region}_{resolution}_{ts}.json"
                
                try:
                    response = requests.get(data_url, timeout=30)
                    response.raise_for_status()
                    data = response.json()
                    
                    # 解析数据
                    series = data.get('series', [])
                    
                    for item in series:
                        if isinstance(item, list) and len(item) >= 2:
                            ts_val, gen_val = item[0], item[1]
                            energy_data.append({
                                'timestamp': ts_val,
                                col_name: gen_val
                            })
                            
                except Exception as e:
                    print(f"    下载失败 {col_name} {ts}: {e}")
                    continue
            
            time.sleep(0.1)  # 短暂延迟避免请求过于频繁
        
        print(f"完成下载 {col_name}: {len(energy_data)} 条记录")
        return energy_data
    
    # 使用线程池并行下载
    all_generation_data = pd.DataFrame()
    
    with ThreadPoolExecutor(max_workers=4) as executor:  # 限制并发数避免被限流
        # 提交所有任务
        future_to_energy = {
            executor.submit(download_energy_type, filter_id, col_name): (filter_id, col_name) 
            for filter_id, col_name in energy_types.items()
        }
        
        # 收集结果
        for future in as_completed(future_to_energy):
            filter_id, col_name = future_to_energy[future]
            try:
                energy_data = future.result()
                if energy_data:
                    df_energy = pd.DataFrame(energy_data)
                    if all_generation_data.empty:
                        all_generation_data = df_energy
                    else:
                        all_generation_data = pd.merge(all_generation_data, df_energy, on='timestamp', how='outer')
                    print(f"合并完成 {col_name}")
            except Exception as e:
                print(f"处理 {col_name} 时出错: {e}")
    
    # 4. 转换为DataFrame并清理
    if all_generation_data.empty:
        print("未获取到任何数据")
        return None
        
    df = all_generation_data.copy()
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df.sort_values('datetime').reset_index(drop=True)
    
    # 填充缺失值
    numeric_cols = list(energy_types.values())
    df[numeric_cols] = df[numeric_cols].fillna(0)
    
    print(f"\n✅ 成功下载发电数据: {len(df)} 条记录")
    print(f"时间范围: {df['datetime'].min()} 到 {df['datetime'].max()}")
    print(f"列: {list(df.columns)}")
    
    return df

# 使用示例
df_generation = download_actual_generation("2019-01-01", "2026-04-05")
if df_generation is not None:
    # 保存为CSV
    df_generation.to_csv("Actual_generation_20190101_20260405.csv", index=False)
    print("数据已保存到 Actual_generation_20190101_20260405.csv")

开始下载发电数据: 2019-01-01 到 2026-04-05
获取时间戳列表: https://www.smard.de/app/chart_data/4066/DE/index_quarterhour.json
找到 588 个时间戳
筛选出 379 个相关时间戳
时间戳范围: 2018-12-30 23:00:00 到 2026-03-29 22:00:00
开始下载 biomass (filter: 4066)
开始下载 hydropower (filter: 1226)
开始下载 wind_offshore (filter: 1225)
开始下载 wind_onshore (filter: 4067)
完成下载 wind_offshore: 254684 条记录
开始下载 photovoltaics (filter: 4068)
完成下载 hydropower: 254684 条记录
开始下载 other_renewable (filter: 1228)
合并完成 wind_offshore
合并完成 hydropower
完成下载 biomass: 254684 条记录
开始下载 nuclear (filter: 1224)
完成下载 wind_onshore: 254684 条记录
开始下载 lignite (filter: 1223)
合并完成 biomass
合并完成 wind_onshore
    下载失败 nuclear 1707087600000: 404 Client Error: Not Found for url: https://www.smard.de/app/chart_data/1224/DE/1224_DE_quarterhour_1707087600000.json
    下载失败 nuclear 1707692400000: 404 Client Error: Not Found for url: https://www.smard.de/app/chart_data/1224/DE/1224_DE_quarterhour_1707692400000.json
    下载失败 nuclear 1708297200000: 404 Client Error: Not Found for url: https://w

In [ ]:
# 读取CSV文件
df = pd.read_csv('/workspaces/Energy-Monitor-Germany/notebooks/Actual_generation_20190101_20260405.csv')

# 查看前几行
print(df.head())
print(f"\n数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n数据类型:\n{df.dtypes}")

       timestamp  biomass  hydropower  wind_offshore  wind_onshore  \
0  1546815600000  1176.25      348.00         755.25       1899.75   
1  1546816500000  1176.00      342.50         742.50       1948.25   
2  1546817400000  1176.00      341.75         709.50       1986.00   
3  1546818300000  1176.75      340.75         681.00       2043.00   
4  1546819200000  1169.00      363.50         647.75       2089.75   

   photovoltaics  other_renewable  nuclear  lignite  hard_coal  fossil_gas  \
0            0.0            33.00  2349.75  3956.75    2404.75     1624.75   
1            0.0            33.25  2352.75  3996.75    2336.50     1614.00   
2            0.0            33.00  2350.00  3958.50    2284.00     1622.50   
3            0.0            33.00  2344.00  3966.25    2253.25     1620.00   
4            0.0            33.25  2354.00  3984.75    2223.50     1606.25   

   hydro_pumped_storage  other_conventional             datetime  
0                   0.0              430.50

In [10]:
with open("Actual_generation_20190101_20260405.csv") as f:
    print(repr(f.readline()))
    print(repr(f.readline()))

'timestamp,biomass,hydropower,wind_offshore,wind_onshore,photovoltaics,other_renewable,nuclear,lignite,hard_coal,fossil_gas,hydro_pumped_storage,other_conventional,datetime\n'
'1546815600000,1176.25,348.0,755.25,1899.75,0.0,33.0,2349.75,3956.75,2404.75,1624.75,0.0,430.5,2019-01-06 23:00:00\n'


In [12]:
df_gen_api = pd.read_csv("Actual_generation_20190101_20260405.csv")

# datetime 列转成带时区的 UTC
df_gen_api['timestamp_utc'] = pd.to_datetime(
    df_gen_api['datetime']
).dt.tz_localize('UTC')  # API 返回的已经是 UTC

df_gen_api = df_gen_api.drop(columns=['timestamp', 'datetime'])

print(df_gen_api.shape)
print(df_gen_api.dtypes)
print(df_gen_api.head(2))

(254012, 13)
biomass                             float64
hydropower                          float64
wind_offshore                       float64
wind_onshore                        float64
photovoltaics                       float64
other_renewable                     float64
nuclear                             float64
lignite                             float64
hard_coal                           float64
fossil_gas                          float64
hydro_pumped_storage                float64
other_conventional                  float64
timestamp_utc           datetime64[us, UTC]
dtype: object
   biomass  hydropower  wind_offshore  wind_onshore  photovoltaics  \
0  1176.25       348.0         755.25       1899.75            0.0   
1  1176.00       342.5         742.50       1948.25            0.0   

   other_renewable  nuclear  lignite  hard_coal  fossil_gas  \
0            33.00  2349.75  3956.75    2404.75     1624.75   
1            33.25  2352.75  3996.75    2336.50     1614.00   



In [13]:
# API CSV
print(df_gen_api['timestamp_utc'].min())  # 应该是 2019-01-01 附近
print(df_gen_api['timestamp_utc'].max())  # 应该是 2026-04-05

2019-01-06 23:00:00+00:00
2026-04-05 21:45:00+00:00


In [18]:
import requests

# 看 generation 的第一个 timestamp 是什么
r = requests.get("https://www.smard.de/app/chart_data/4068/DE/index_quarterhour.json")
timestamps = r.json()['timestamps']
import pandas as pd
print(pd.to_datetime(timestamps[0], unit='ms', utc=True))
print(pd.to_datetime(timestamps[1], unit='ms', utc=True))

2014-12-28 23:00:00+00:00
2015-01-04 23:00:00+00:00
